# Choosing where to place apertures for stellar photometry

In [ ]:

import numpy as np
from astropy.nddata import CCDData, Cutout2D
from photutils.centroids import centroid_1dg, centroid_2dg, centroid_com, centroid_quadratic

%matplotlib widget
from matplotlib import pyplot as plt

plt.style.use('../photutils_notebook_style.mplstyle')

In [ ]:
from astropy.visualization import AsymmetricPercentileInterval

In [ ]:
ccd = CCDData.read("TIC_125489084.01-S001-R055-C001-ip.fit")

In [ ]:
plt.figure()
inter = AsymmetricPercentileInterval(lower_percentile=40, upper_percentile=95)
plt.imshow(inter(ccd.data), origin="lower")
plt.grid()
plt.show()

In [ ]:
lower_right_chunk = ccd.data[50:550, 3500:]
plt.figure()
plt.imshow(inter(lower_right_chunk), origin="lower")
plt.grid()
plt.show()

In [ ]:
rough_centers = {
    1: (290, 240),  # KEEP
    2: (180, 340),  # KEEP
    3: (80, 40),
    4: (525, 230),
}

centroid_region_width = 50

centroids = {}
for centroid_method in [centroid_1dg, centroid_2dg, centroid_com, centroid_quadratic]:
    centroids[centroid_method.__name__] = {}
    for star, center in rough_centers.items():
        mini_cut = Cutout2D(lower_right_chunk, center, centroid_region_width)
        mini_data = mini_cut.data - np.median(mini_cut.data)
        tmp = centroid_method(mini_data)
        centroids[centroid_method.__name__][star] = mini_cut.to_original_position(tmp)



# mini_data = ccd.data[255:305, 4005:4055]
# mini_data = mini_data - np.median(mini_data)

# centroids = {}


In [ ]:
plt.figure()
inter = AsymmetricPercentileInterval(lower_percentile=40, upper_percentile=95)
plt.imshow(inter(lower_right_chunk), origin="lower", cmap="gray", alpha=0.25)

for name, locs in centroids.items():
    plt.scatter([c[0] for c in locs.values()], [c[1] for c in locs.values()], marker="+", label=name)

ax = plt.gca()
extent = (0, lower_right_chunk.shape[0], 0, lower_right_chunk.shape[1])
zoomfac = 3

inset_locs = (100, 300)
cutout_factor = (1.5, 10)
for loc, center, cutout in zip(inset_locs, rough_centers.values(), cutout_factor):
    axins = ax.inset_axes(
        (loc, loc, zoomfac * centroid_region_width, zoomfac * centroid_region_width),  # where to put the inset and its size
        transform=ax.transData,  # use the same coordinate system as the main plot
        xlim=(center[0] - centroid_region_width/cutout, center[0] + centroid_region_width/cutout),  # limits of the inset
        ylim=(center[1] - centroid_region_width/cutout, center[1] + centroid_region_width/cutout),
        xticks=[],  # turn off ticks
        yticks=[],
        )
    ax.indicate_inset_zoom(axins, edgecolor="red")
    axins.imshow(inter(lower_right_chunk), origin="lower", cmap="gray", alpha=0.5)
    for name, locs in centroids.items():
        axins.scatter([c[0] for c in locs.values()], [c[1] for c in locs.values()], marker="+", label=name)

plt.legend(loc="lower left", bbox_to_anchor=(0.05, 1.05), ncols=2)
plt.tight_layout()
plt.show()

In [ ]:
centroids